Dieses Notebook:

- lädt `tiktok-dataset-real.csv` und `tiktok-dataset-ai.csv`
- fügt eine Spalte `category` ("real" / "ai") hinzu
- definiert einen `influencer`-Identifier
- erstellt eine Übersicht der Anzahl Videos und des Engagements pro Influencer und Kategorie

---

In [1]:
# Influencer-balancierte Stichprobe mit Engagement-Schichtung
# Zeitstempel fr?h parsen
# 50 Videos pro Influencerin ziehen
# Low-, Medium- und High-Engagement abdecken

import pandas as pd

INPUT_CSV = "../01_raw/videos_metadata/01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv"
OUT_CSV = "../03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv"
LOG_CSV = "../03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50_LOG.csv"

INFL_COL = "influencer"
ENG_COL = "video_engagement_rate"
TS_COL = "video_timestamp"
TARGET = 50
BINS = ["Low", "Medium", "High"]
SEED = 42

# Vollst?ndigen Videodatensatz laden
df = pd.read_csv(INPUT_CSV)

# Zeitstempel f?r sp?tere Datumsoperationen parsen
df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce")

df[ENG_COL] = pd.to_numeric(df[ENG_COL], errors="coerce")
df = df.dropna(subset=[INFL_COL, ENG_COL]).copy()


In [2]:
# Engagement-Bins pro Influencerin bilden
def add_bins(g):
    r = g[ENG_COL].rank(method="first", pct=True)
    g = g.copy()
    g["eng_bin"] = pd.cut(r, bins=[0, 1/3, 2/3, 1], labels=BINS, include_lowest=True)
    return g

df = df.groupby(INFL_COL, group_keys=False).apply(add_bins).reset_index(drop=True)

# Zielverteilung pro Bin festlegen
def plan_counts(n_total):
    base, rest = divmod(n_total, 3)
    counts = [base, base, base]
    for i in [0, 2, 1][:rest]:
        counts[i] += 1
    return dict(zip(BINS, counts))

# Pro Influencerin sampeln und fehlende F?lle aus ?brigen Bins erg?nzen

def sample_one(g, target=50, seed=42):
    n = min(target, len(g))
    plan = plan_counts(n)

    avail = {b: int((g["eng_bin"] == b).sum()) for b in BINS}
    take = {b: min(plan[b], avail[b]) for b in BINS}

    missing = n - sum(take.values())
    while missing > 0:
        free = {b: avail[b] - take[b] for b in BINS}
        candidates = [b for b in BINS if free[b] > 0]
        if not candidates:
            break
        b = max(candidates, key=lambda x: free[x])
        take[b] += 1
        missing -= 1

    parts = []
    for i, b in enumerate(BINS):
        k = take[b]
        if k > 0:
            parts.append(g[g["eng_bin"] == b].sample(n=k, random_state=seed + i))
    sampled = pd.concat(parts, ignore_index=True) if parts else g.iloc[0:0].copy()

    log = {
        "influencer": g[INFL_COL].iloc[0],
        "available": len(g),
        "target": target,
        "sampled": len(sampled),
        "planned_low": plan["Low"],
        "planned_medium": plan["Medium"],
        "planned_high": plan["High"],
        "taken_low": take["Low"],
        "taken_medium": take["Medium"],
        "taken_high": take["High"],
    }
    return sampled, log

sampled_parts, logs = [], []
for i, (_, g) in enumerate(df.groupby(INFL_COL)):
    s, l = sample_one(g, target=TARGET, seed=SEED + i * 10)
    sampled_parts.append(s)
    logs.append(l)

sampled = pd.concat(sampled_parts, ignore_index=True)
log_df = pd.DataFrame(logs)

sampled.to_csv(OUT_CSV, index=False)
log_df.to_csv(LOG_CSV, index=False)

C:\Users\hanna\AppData\Local\Temp\ipykernel_6172\2690058132.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(INFL_COL, group_keys=False).apply(add_bins).reset_index(drop=True)


In [3]:
# Plausibilit?tschecks
print("Videos per influencer (should be ~50):")
print(sampled.groupby(INFL_COL).size().describe())

print("Bin distribution per influencer:")
bins_pivot = (
    sampled.groupby([INFL_COL, "eng_bin"], observed=False)
           .size()
           .reset_index(name="n_videos")
           .pivot_table(index=INFL_COL, columns="eng_bin", values="n_videos", fill_value=0)
           .reindex(columns=BINS)
)
print(bins_pivot)

Videos per influencer (should be ~50):
count    10.0
mean     50.0
std       0.0
min      50.0
25%      50.0
50%      50.0
75%      50.0
max      50.0
dtype: float64
Bin distribution per influencer:
eng_bin               Low  Medium  High
influencer                             
ai.kalai             17.0    16.0  17.0
brexleyai            17.0    16.0  17.0
ericanic0le          17.0    16.0  17.0
hollybrandmusic      17.0    16.0  17.0
imma.tokyo           16.0    17.0  17.0
jessicawangofficial  17.0    16.0  17.0
lilmiquela           17.0    16.0  17.0
millas_sofia         17.0    16.0  17.0
misshannahashton     17.0    16.0  17.0
nobodywhoareu        17.0    16.0  17.0


C:\Users\hanna\AppData\Local\Temp\ipykernel_6172\2537777565.py:10: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(index=INFL_COL, columns="eng_bin", values="n_videos", fill_value=0)


In [5]:
# Bereits gezogene Videos als Filterbasis verwenden.
videos = pd.read_csv("../03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv")
REAL_COMMENTS_PATH = "../01_raw/comments_metadata/02_REAL_TIKTOK_COMMENTS_DATASET_2025.csv"
AI_COMMENTS_PATH = "../01_raw/comments_metadata/02_AI_TIKTOK_COMMENTS_DATASET_2025.csv"

In [6]:
df_real_c = pd.read_csv(REAL_COMMENTS_PATH)
df_ai_c   = pd.read_csv(AI_COMMENTS_PATH)

df_real_c["influencer_type"] = "real"
df_ai_c["influencer_type"]   = "ai"

df_comments = pd.concat([df_real_c, df_ai_c], ignore_index=True)
print(df_comments.shape)
df_comments.head()


(84419, 20)


,id,thread_id,author,author_full,author_avatar_url,body,timestamp,unix_timestamp,likes,replies,post_id,post_url,post_body,comment_url,is_liked_by_post_author,is_sticky,is_comment_on_comment,language_guess,missing_fields,influencer_type
0,7040985769041330950,7040984071631228206,rolandobendeck,rolando bendeck,https://p77-sign-va.tiktokcdn.com/tos-maliva-a...,So y’all rich rich. Huh. Are you looking to ad...,2021-12-13 01:00:55,1639357255,34876,32,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
1,7068917289810625286,7040984071631228206,vassosofou,Vasso Sofou,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,Is there a balcony?? I couldn't live without air!,2022-02-26 07:29:31,1645860571,125,6,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
2,7041010695568507695,7040984071631228206,mommymelon_64,S. Gamer,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,"30,000/month?",2021-12-13 02:37:41,1639363061,298,6,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
3,7041018872992137986,7040984071631228206,cowmoo0,James Madison,https://p77-sign-sg.tiktokcdn.com/tos-alisg-av...,You need a fire alarm? I can scream,2021-12-13 03:09:30,1639364970,8001,22,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
4,7043572331746296582,7040984071631228206,imjuliadali,Nina Voro,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,What is this trend!?? Why bragging,2021-12-20 00:18:03,1639959483,11,2,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real


In [7]:
df_comments = df_comments.drop_duplicates(subset=["id"])
print("Comments after dedupe:", df_comments.shape)

Comments after dedupe: (80817, 20)


Erstes samplen der kommentare, nicht nutzbare Kommentare wegschmeißen

In [8]:
import re

comments_f = df_comments.dropna(subset=["body"]).copy()
comments_f["body"] = comments_f["body"].astype(str).str.strip()

# Leere Texte entfernen
comments_f = comments_f[comments_f["body"] != ""]
# Sprachfilter
comments_f = comments_f[comments_f["language_guess"].isin(["en", "unknown"])].copy()

# Reine Links, Mentions, Hashtags und Emoji-Kommentare entfernen
def is_usable(t: str) -> bool:
    t = t.strip()
    if t == "":
        return False
    # Nur Links
    if re.fullmatch(r"(https?://\S+)", t):
        return False
    # Nur Mentions oder Hashtags
    if re.fullmatch(r"[@#]\w+(?:\s+[@#]\w+)*", t):
        return False
    # Buchstaben und Zahlen entfernen und Rest pr?fen
    alnum = re.sub(r"[^\w]", "", t, flags=re.UNICODE)
    if len(alnum) == 0:
        return False
    return True

comments_f = comments_f[comments_f["body"].apply(is_usable)].copy()

print("After filters:", comments_f.shape)
comments_f.head()


After filters: (39579, 20)


,id,thread_id,author,author_full,author_avatar_url,body,timestamp,unix_timestamp,likes,replies,post_id,post_url,post_body,comment_url,is_liked_by_post_author,is_sticky,is_comment_on_comment,language_guess,missing_fields,influencer_type
0,7040985769041330950,7040984071631228206,rolandobendeck,rolando bendeck,https://p77-sign-va.tiktokcdn.com/tos-maliva-a...,So y’all rich rich. Huh. Are you looking to ad...,2021-12-13 01:00:55,1639357255,34876,32,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
1,7068917289810625286,7040984071631228206,vassosofou,Vasso Sofou,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,Is there a balcony?? I couldn't live without air!,2022-02-26 07:29:31,1645860571,125,6,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
2,7041010695568507695,7040984071631228206,mommymelon_64,S. Gamer,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,"30,000/month?",2021-12-13 02:37:41,1639363061,298,6,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
3,7041018872992137986,7040984071631228206,cowmoo0,James Madison,https://p77-sign-sg.tiktokcdn.com/tos-alisg-av...,You need a fire alarm? I can scream,2021-12-13 03:09:30,1639364970,8001,22,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real
4,7043572331746296582,7040984071631228206,imjuliadali,Nina Voro,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,What is this trend!?? Why bragging,2021-12-20 00:18:03,1639959483,11,2,7040984071631228206,https://m.tiktok.com/v/7040984071631228206,What do you guys think of the view? #nycliving...,https://m.tiktok.com/v/7040984071631228206.htm...,no,no,no,en,NaN,real


In [9]:
# Sicherstellen, dass Join-Spalten vergleichbar sind (oft Strings)
comments_f["post_id"] = comments_f["post_id"].astype(str)
videos["video_id"] = videos["video_id"].astype(str)

# Metadaten f?r den Join
meta_cols = [c for c in ["influencer", "video_timestamp", "video_view_count", "video_like_count", "video_comment_count", "video_share_count", "video_engagement_rate"] if c in videos.columns]
meta = videos[["video_id"] + meta_cols].drop_duplicates(subset=["video_id"])
comments_enriched = comments_f.merge(
    meta,
    left_on="post_id",
    right_on="video_id",
    how="inner"  # nur Kommentare zu Videos aus dem Sample
)

print("Comments after join to sampled videos:", comments_enriched.shape)
comments_enriched.head()

Comments after join to sampled videos: (17030, 28)


,id,thread_id,author,author_full,author_avatar_url,body,timestamp,unix_timestamp,likes,replies,...,missing_fields,influencer_type,video_id,influencer,video_timestamp,video_view_count,video_like_count,video_comment_count,video_share_count,video_engagement_rate
0,7572978399367676686,7572978253188711181,yourelovedby_sally,☆ST4R ☆ saw enha 8/7,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,and the little sister too,2025-11-15 15:44:17,1763221457,1,0,...,NaN,real,7571506841398562078,jessicawangofficial,2025-11-11 16:33:55,9348,412,15,6,0.046320
1,7573676065903493896,7539911944354958606,kobebryan168,kobiee🕷,https://p16-sign-sg.tiktokcdn.com/tos-alisg-av...,where you from,2025-11-17 12:51:25,1763383885,0,0,...,NaN,real,7539911944354958606,jessicawangofficial,2025-08-18 13:09:25,24000,394,24,6,0.017667
2,7572406827230413586,7539911944354958606,halloween.candies,🕷️🕸️🎃Halloween Candies🍬🧡,https://p16-sign-sg.tiktokcdn.com/tos-alisg-av...,Is anyond allowed to go there or only celebrities,2025-11-14 02:46:09,1763088369,1,0,...,NaN,real,7539911944354958606,jessicawangofficial,2025-08-18 13:09:25,24000,394,24,6,0.017667
3,7540093122681864991,7539911944354958606,guess.the.name.of5,Coraline🗝️🐈‍⬛🟡,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,Early,2025-08-19 00:52:45,1755564765,1,0,...,NaN,real,7539911944354958606,jessicawangofficial,2025-08-18 13:09:25,24000,394,24,6,0.017667
4,7539912856344167175,7539911944354958606,kattastic_444,kat4ever❤💕,https://p77-sign-sg-lite.tiktokcdn.com/tos-ali...,YEY first,2025-08-18 13:12:49,1755522769,1,0,...,NaN,real,7539911944354958606,jessicawangofficial,2025-08-18 13:09:25,24000,394,24,6,0.017667


In [10]:
counts_influencer = (
    comments_enriched
    .groupby(["influencer_type", "influencer"])
    .size()
    .reset_index(name="n_comments")
    .sort_values(["influencer_type", "n_comments"], ascending=[True, False])
)

counts_type = comments_enriched["influencer_type"].value_counts().reset_index()
counts_type.columns = ["influencer_type", "n_comments"]

print(counts_type)
print(counts_influencer)

  influencer_type  n_comments
0              ai       10658
1            real        6372
  influencer_type           influencer  n_comments
3              ai           lilmiquela        7198
2              ai           imma.tokyo        1428
0              ai             ai.kalai         801
4              ai         millas_sofia         783
1              ai            brexleyai         448
9            real        nobodywhoareu        2994
5            real          ericanic0le        2443
8            real     misshannahashton         668
6            real      hollybrandmusic         140
7            real  jessicawangofficial         127


In [11]:
comments_per_video = (
    comments_enriched
    .groupby(["influencer_type", "influencer", "post_id"])
    .size()
    .reset_index(name="n_comments_on_video")
    .sort_values("n_comments_on_video", ascending=False)
)

comments_per_video.head(20)


,influencer_type,influencer,post_id,n_comments_on_video
127,ai,lilmiquela,7129589619099847982,1941
364,real,nobodywhoareu,7586650207677418766,1656
112,ai,lilmiquela,7027954993026190598,1450
117,ai,lilmiquela,7068736017146268974,1117
111,ai,lilmiquela,6988896764241693958,627
82,ai,imma.tokyo,7091167548951039234,498
216,real,ericanic0le,7577086972745813262,405
221,real,ericanic0le,7588992707582758158,394
3,ai,ai.kalai,7515933454010191150,384
175,real,ericanic0le,7459923178039856430,376


In [12]:
comments_enriched.groupby(["influencer_type", "influencer"]).size()


influencer_type  influencer         
ai               ai.kalai                801
                 brexleyai               448
                 imma.tokyo             1428
                 lilmiquela             7198
                 millas_sofia            783
real             ericanic0le            2443
                 hollybrandmusic         140
                 jessicawangofficial     127
                 misshannahashton        668
                 nobodywhoareu          2994
dtype: int64

In [13]:
rename_map = {
    # Kommentare
    "id": "comment_id",
    "thread_id": "comment_thread_id",
    "author": "comment_author_username",
    "author_full": "comment_author_displayname",
    "author_avatar_url": "comment_author_avatar_url",
    "body": "comment_text",
    "timestamp": "comment_timestamp",
    "unix_timestamp": "comment_unix_timestamp",
    "likes": "comment_like_count",
    "replies": "comment_reply_count",
    "post_id": "video_id",
    "post_url": "video_url",
    "post_body": "video_caption",
    "comment_url": "comment_url",
    "is_liked_by_post_author": "comment_liked_by_creator",
    "is_sticky": "comment_is_pinned",
    "is_comment_on_comment": "is_reply_to_comment",
    "language_guess": "comment_language",
    "missing_fields": "comment_missing_fields",
    "influencer_type": "influencer_type",

    # Videos
    "video_id": "video_internal_id",
    "influencer": "influencer",
    "video_timestamp": "video_timestamp",
    "video_view_count": "video_view_count",
    "video_like_count": "video_like_count",
    "video_comment_count": "video_comment_count",
    "video_share_count": "video_share_count",
    "video_engagement_rate": "video_engagement_rate",
}

df_comments = df_comments.rename(columns=rename_map)    
comments_enriched_renamed = comments_enriched.rename(columns=rename_map)

comments_enriched_renamed.columns

Index(['comment_id', 'comment_thread_id', 'comment_author_username',
       'comment_author_displayname', 'comment_author_avatar_url',
       'comment_text', 'comment_timestamp', 'comment_unix_timestamp',
       'comment_like_count', 'comment_reply_count', 'video_id', 'video_url',
       'video_caption', 'comment_url', 'comment_liked_by_creator',
       'comment_is_pinned', 'is_reply_to_comment', 'comment_language',
       'comment_missing_fields', 'influencer_type', 'video_internal_id',
       'influencer', 'video_timestamp', 'video_view_count', 'video_like_count',
       'video_comment_count', 'video_share_count', 'video_engagement_rate'],
      dtype='object')

In [14]:
# Alle Kommentare (ohne Filter) speichern
df_comments.to_csv(
    "../01_raw/comments_metadata/02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv",
    index=False
)

# Gefilterte + angereicherte Kommentare speichern
comments_enriched_renamed.to_csv(
    "../03_datasets/influencer_balanced/02_AI_AND_REAL_TIKTOK_COMMENTS_FOR_STRATIFIED_VIDEOS.csv",
    index=False
)

print("Gespeichert: 02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv und 02_AI_AND_REAL_TIKTOK_COMMENTS_FOR_STRATIFIED_VIDEOS.csv")

Gespeichert: 02_AI_AND_REAL_TIKTOK_COMMENTS_DATASET_2025.csv und 02_AI_AND_REAL_TIKTOK_COMMENTS_FOR_STRATIFIED_VIDEOS.csv


In [20]:
import os
import shutil
import pandas as pd

# Eingabe-CSVs
CSV_INFLUENCER_BALANCED = "../03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv"

# Quellordner
# Getrennte Quellordner
SRC_AI_VIDEOS   = "../02_media/ai_videos_2025/videos"
SRC_REAL_VIDEOS = "../02_media/real_videos_2025/videos"
SRC_AI_FRAMES   = "../02_media/ai_videos_2025/frames"
SRC_REAL_FRAMES = "../02_media/real_videos_2025/frames"

# Zielordner
OUT_VIDEOS = "../02_media/stratified_sample/videos"
OUT_FRAMES = "../02_media/stratified_sample/frames"

os.makedirs(OUT_VIDEOS, exist_ok=True)
os.makedirs(OUT_FRAMES, exist_ok=True)

# Video-ID-Spalte finden
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

# CSVs laden und Video-IDs zusammenf?hren
df_a = pd.read_csv(CSV_INFLUENCER_BALANCED)

id_col_a = find_col(df_a, ["video_id", "id", "post_id", "aweme_id", "item_id"])

assert id_col_a is not None, "Could not find video id column in influencer-balanced CSV."


ids_a = set(df_a[id_col_a].astype(str))


print("=== EXPECTED VIDEO COUNTS ===")
print(f"Influencer-balanced videos: {len(ids_a)}")

# Kopierlogik
def copy_video_file(vid: str) -> bool:
    """Try to copy vid.mp4 from available source folders into OUT_VIDEOS. Returns True if copied."""
    dst = os.path.join(OUT_VIDEOS, f"{vid}.mp4")
    if os.path.exists(dst):
        return True  # bereits vorhanden
    
    candidates = []
    candidates += [
        os.path.join(SRC_AI_VIDEOS, f"{vid}.mp4"),
        os.path.join(SRC_REAL_VIDEOS, f"{vid}.mp4"),
    ]
    
    for src in candidates:
        if src and os.path.exists(src):
            shutil.copy2(src, dst)
            return True
    return False

def copy_frame_folder(vid: str) -> bool:
    """Try to copy frame folder named vid into OUT_FRAMES. Returns True if copied."""
    dst = os.path.join(OUT_FRAMES, vid)
    if os.path.exists(dst):
        return True  # bereits vorhanden
    
    candidates = []
    candidates += [
        os.path.join(SRC_AI_FRAMES, vid),
        os.path.join(SRC_REAL_FRAMES, vid),
    ]
    
    for src in candidates:
        if src and os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            return True
    return False

# Kopieren ausf?hren
missing_videos = []
missing_frames = []

for vid in sorted(ids_a):
    if not copy_video_file(vid):
        missing_videos.append(vid)
    if not copy_frame_folder(vid):
        missing_frames.append(vid)

# Abschlusschecks
copied_video_files = [
    f for f in os.listdir(OUT_VIDEOS) if f.endswith(".mp4")
]
copied_frame_dirs = [
    d for d in os.listdir(OUT_FRAMES)
    if os.path.isdir(os.path.join(OUT_FRAMES, d))
]

print("\n=== COPY RESULTS ===")
print(f"Videos expected: {len(ids_a)}")
print(f"Videos copied:   {len(copied_video_files)}")
print(f"Frames copied:   {len(copied_frame_dirs)}")

# Details zu fehlenden Dateien
if missing_videos:
    print(f"\nHinweis: Missing VIDEO files ({len(missing_videos)}):")
    print(missing_videos[:10], "..." if len(missing_videos) > 10 else "")
else:
    print("\nAll videos found.")

if missing_frames:
    print(f"\nHinweis: Missing FRAME folders ({len(missing_frames)}):")
    print(missing_frames[:10], "..." if len(missing_frames) > 10 else "")
else:
    print("\nAll frame folders found.")

=== EXPECTED VIDEO COUNTS ===
Influencer-balanced videos: 500

=== COPY RESULTS ===
Videos expected: 500
Videos copied:   0
Frames copied:   0

Hinweis: Missing VIDEO files (500):
['6915681317556440321', '6929732205350538497', '6931646908108672258', '6970260794692947201', '6988079288910220545', '6988896764241693958', '6990687664890563841', '7001695430295833858', '7006386742462811398', '7011606995383979269'] ...

Hinweis: Missing FRAME folders (500):
['6915681317556440321', '6929732205350538497', '6931646908108672258', '6970260794692947201', '6988079288910220545', '6988896764241693958', '6990687664890563841', '7001695430295833858', '7006386742462811398', '7011606995383979269'] ...


In [22]:
import shutil
from pathlib import Path
import pandas as pd

# Fehlende Dateien f?r die stratifizierte Stichprobe kopieren
DATA_DIR = Path('..')
CSV_STRATIFIED = DATA_DIR / '03_datasets' / 'influencer_balanced' / '01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUT_FRAMES_STRATIFIED = DATA_DIR / '02_media/stratified_sample/frames'
OUT_VIDEOS_STRATIFIED = DATA_DIR / '02_media/stratified_sample/videos'

# Nur 2025-Quellordner verwenden
AI_FRAMES_ROOT = DATA_DIR / '02_media/ai_videos_2025/frames'
REAL_FRAMES_ROOT = DATA_DIR / '02_media/real_videos_2025/frames'
AI_VIDEOS_ROOT = DATA_DIR / '02_media/ai_videos_2025/videos'
REAL_VIDEOS_ROOT = DATA_DIR / '02_media/real_videos_2025/videos'

for p in [AI_FRAMES_ROOT, REAL_FRAMES_ROOT, AI_VIDEOS_ROOT, REAL_VIDEOS_ROOT]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required source folder: {p}')

videos = pd.read_csv(CSV_STRATIFIED)
video_id_col = 'video_id' if 'video_id' in videos.columns else 'id'
video_ids = sorted(set(videos[video_id_col].astype(str)))

OUT_FRAMES_STRATIFIED.mkdir(parents=True, exist_ok=True)
OUT_VIDEOS_STRATIFIED.mkdir(parents=True, exist_ok=True)

frames_copied = 0
frames_already = 0
frames_missing = []
videos_copied = 0
videos_already = 0
videos_missing = []

for vid in video_ids:
    # Frames nur kopieren, wenn der Zielordner fehlt
    dst_frames = OUT_FRAMES_STRATIFIED / vid
    if dst_frames.exists():
        frames_already += 1
    else:
        src_ai_frames = AI_FRAMES_ROOT / vid
        src_real_frames = REAL_FRAMES_ROOT / vid

        if src_ai_frames.is_dir():
            shutil.copytree(src_ai_frames, dst_frames)
            frames_copied += 1
        elif src_real_frames.is_dir():
            shutil.copytree(src_real_frames, dst_frames)
            frames_copied += 1
        else:
            frames_missing.append(vid)

    # Videos nur kopieren, wenn die Zieldatei fehlt
    dst_video = OUT_VIDEOS_STRATIFIED / f'{vid}.mp4'
    if dst_video.exists():
        videos_already += 1
    else:
        src_ai_video = AI_VIDEOS_ROOT / f'{vid}.mp4'
        src_real_video = REAL_VIDEOS_ROOT / f'{vid}.mp4'

        if src_ai_video.is_file():
            shutil.copy2(src_ai_video, dst_video)
            videos_copied += 1
        elif src_real_video.is_file():
            shutil.copy2(src_real_video, dst_video)
            videos_copied += 1
        else:
            videos_missing.append(vid)

print('=== STRATIFIED INCREMENTAL COPY SUMMARY ===')
print(f'CSV videos:               {len(video_ids)}')
print('--- Frames ---')
print(f'Copied now:               {frames_copied}')
print(f'Already in target:        {frames_already}')
print(f'Missing frame folder:     {len(frames_missing)}')
if frames_missing:
    print('First missing frame IDs: ', frames_missing[:20])

print('--- Videos ---')
print(f'Copied now:               {videos_copied}')
print(f'Already in target:        {videos_already}')
print(f'Missing video files:      {len(videos_missing)}')
if videos_missing:
    print('First missing video IDs: ', videos_missing[:20])



=== STRATIFIED INCREMENTAL COPY SUMMARY ===
CSV videos:               500
--- Frames ---
Copied now:               134
Already in target:        366
Missing frame folder:     0
--- Videos ---
Copied now:               500
Already in target:        0
Missing video files:      0
